In [ ]:
# ============================================================
# CELL 0 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

print("=" * 80)
print("GOOGLE DRIVE CHECK")
print("=" * 80)
print("ROOT:", ROOT)
print("Exists:", ROOT.exists())

if ROOT.exists():
    print("✓ Google Drive mounted")
    print("✓ COST_PROJECT folder found")
else:
    raise FileNotFoundError(
        f"Folder not found: {ROOT}\n"
        "Check the Google Drive account used by this Colab."
    )

In [ ]:
# ============================================================
# COST REVIEWER 2 — FINAL AUDITS
# CELL 1 — ENVIRONMENT + FROZEN COST ARTIFACT DISCOVERY
#
# PURPOSE:
#   Prepare a completely separate reviewer-audit notebook.
#
# IMPORTANT:
#   - NO TRAINING
#   - NO FINE-TUNING
#   - NO NEW EXTERNAL VALIDATION
#   - NO MODIFICATION OF EXISTING RESULTS
#   - NO NEW NORMALIZATION FITTING
#
# This notebook will only address remaining reviewer audits:
#   R2-C3 : sparsity distribution
#   R2-C4 : FLOPs / computational characteristics
#   R2-C5 : statistical traceability
#   R2-C6 : patient-level leakage / cohort audit
# ============================================================

import os
import json
import math
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

print("=" * 100)
print("COST REVIEWER 2 — FINAL AUDITS")
print("CELL 1 — ENVIRONMENT + FROZEN ARTIFACT DISCOVERY")
print("=" * 100)


# ============================================================
# 1. PATHS
# ============================================================

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

CHECKPOINT = (
    ROOT
    / "mimic-iv-2.1"
    / "saved_models"
    / "dsrnn_v2_best.pt"
)

print("\nROOT:")
print(ROOT)

print("\nFrozen COST Improved checkpoint:")
print(CHECKPOINT)

assert ROOT.exists(), (
    f"ROOT directory not found:\n{ROOT}"
)

assert CHECKPOINT.exists(), (
    f"Checkpoint not found:\n{CHECKPOINT}"
)

print(
    f"\n✓ Checkpoint exists "
    f"({CHECKPOINT.stat().st_size / 1024**2:.2f} MB)"
)


# ============================================================
# 2. DEVICE
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", DEVICE)

if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))


# ============================================================
# 3. LOAD FROZEN CHECKPOINT
# ============================================================

print("\n" + "=" * 100)
print("LOADING FROZEN COST IMPROVED CHECKPOINT")
print("=" * 100)

ckpt = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

assert isinstance(ckpt, dict)

print("\nCheckpoint keys:")

for key in ckpt.keys():
    obj = ckpt[key]

    if isinstance(obj, dict):
        print(
            f"  {key:30s} "
            f"type=dict | entries={len(obj)}"
        )
    else:
        print(
            f"  {key:30s} "
            f"type={type(obj).__name__}"
        )


# ============================================================
# 4. VERIFY EXPECTED CHECKPOINT CONTENT
# ============================================================

required_keys = [
    "model_state_dict",
    "config",
    "norm_stats",
]

for key in required_keys:
    assert key in ckpt, (
        f"Required checkpoint key missing: {key}"
    )

state = ckpt["model_state_dict"]
cfg   = ckpt["config"]
norm  = ckpt["norm_stats"]

print("\n✓ Required checkpoint objects found")


# ============================================================
# 5. CHECKPOINT ARCHITECTURE AUDIT
# ============================================================

print("\n" + "=" * 100)
print("FROZEN MODEL ARCHITECTURE AUDIT")
print("=" * 100)

input_weight = state.get("input_proj.0.weight")
static_weight = state.get("fusion.static_proj.weight")

assert input_weight is not None
assert static_weight is not None

n_temporal_input = input_weight.shape[1]
hidden_size      = input_weight.shape[0]
n_static         = static_weight.shape[1]

print(
    f"\nTemporal input dimension : {n_temporal_input}"
)
print(
    f"Static input dimension   : {n_static}"
)
print(
    f"Hidden dimension         : {hidden_size}"
)

assert n_temporal_input == 93
assert n_static == 11

print("\n✓ Expected COST Improved dimensions verified")
print("✓ Temporal input = 93")
print("✓ Static input   = 11")


# ============================================================
# 6. CHECK CONFIGURATION
# ============================================================

print("\n" + "=" * 100)
print("FROZEN CONFIGURATION")
print("=" * 100)

important_config = [
    "hidden_size",
    "n_lstm_layers",
    "n_heads",
    "attention_dim",
    "salience_hidden",
    "dropout",
    "sparsity_tau",
    "lambda_sparse",
    "batch_size",
    "seq_len",
]

for key in important_config:

    value = cfg.get(key, "<NOT STORED>")

    print(
        f"{key:25s}: {value}"
    )


# ============================================================
# 7. VERIFY NORMALIZATION STATISTICS
# ============================================================

print("\n" + "=" * 100)
print("NORMALIZATION ARTIFACT AUDIT")
print("=" * 100)

expected_norm = [
    "feat_mean",
    "feat_std",
    "stat_mean",
    "stat_std",
]

for key in expected_norm:

    assert key in norm, (
        f"Missing normalization object: {key}"
    )

    arr = np.asarray(norm[key])

    print(
        f"{key:15s}: "
        f"shape={arr.shape}"
    )

assert np.asarray(norm["feat_mean"]).shape == (31,)
assert np.asarray(norm["feat_std"]).shape  == (31,)
assert np.asarray(norm["stat_mean"]).shape == (11,)
assert np.asarray(norm["stat_std"]).shape  == (11,)

print("\n✓ Frozen MIMIC-IV normalization artifacts verified")


# ============================================================
# 8. STORED PERFORMANCE
# ============================================================

print("\n" + "=" * 100)
print("STORED COST IMPROVED PERFORMANCE")
print("=" * 100)

for key in [
    "val_auc",
    "test_auc_single",
    "test_auc_ensemble",
]:

    if key in ckpt:

        try:
            value = float(ckpt[key])
            print(
                f"{key:25s}: {value:.6f}"
            )

        except Exception:
            print(
                f"{key:25s}: {ckpt[key]}"
            )


# ============================================================
# 9. SEARCH EXISTING REVIEWER-RELEVANT ARTIFACTS
#
# We are NOT loading or modifying them yet.
# This simply tells us what already exists.
# ============================================================

print("\n" + "=" * 100)
print("SEARCHING EXISTING COST ARTIFACTS")
print("=" * 100)

extensions = {
    ".npy",
    ".npz",
    ".csv",
    ".pkl",
    ".pickle",
    ".pt",
    ".pth",
}

keywords = [
    "prediction",
    "pred",
    "spars",
    "gate",
    "gating",
    "mask",
    "attention",
    "salience",
    "ablation",
    "delong",
    "bootstrap",
    "latency",
    "flop",
    "split",
    "train",
    "val",
    "test",
]

artifact_hits = []

for path in ROOT.rglob("*"):

    if not path.is_file():
        continue

    if path.suffix.lower() not in extensions:
        continue

    name_lower = path.name.lower()

    matched = [
        k for k in keywords
        if k in name_lower
    ]

    if matched:

        artifact_hits.append(
            {
                "path": str(path),
                "filename": path.name,
                "extension": path.suffix.lower(),
                "matched_keywords": ", ".join(matched),
                "size_mb": path.stat().st_size / 1024**2,
            }
        )


artifact_df = pd.DataFrame(artifact_hits)

if len(artifact_df) > 0:

    artifact_df = (
        artifact_df
        .sort_values(
            ["matched_keywords", "filename"]
        )
        .reset_index(drop=True)
    )

    print(
        f"\nCandidate reviewer-audit artifacts found: "
        f"{len(artifact_df):,}"
    )

    pd.set_option(
        "display.max_colwidth",
        120
    )

    display(
        artifact_df[
            [
                "filename",
                "matched_keywords",
                "size_mb",
                "path",
            ]
        ].head(100)
    )

else:

    print(
        "\nNo candidate artifact files found "
        "from filename search."
    )


# ============================================================
# 10. SEARCH COST NOTEBOOKS
# ============================================================

print("\n" + "=" * 100)
print("COST NOTEBOOK INVENTORY")
print("=" * 100)

notebook_keywords = [
    "cost",
    "ds_rnn",
    "dsrnn",
    "revision",
    "gating",
    "improved",
]

notebooks = []

for path in ROOT.rglob("*.ipynb"):

    name_lower = path.name.lower()

    if any(
        k in name_lower
        for k in notebook_keywords
    ):

        notebooks.append(
            {
                "filename": path.name,
                "path": str(path),
                "size_mb": path.stat().st_size / 1024**2,
            }
        )

notebook_df = pd.DataFrame(notebooks)

if len(notebook_df) > 0:

    notebook_df = (
        notebook_df
        .sort_values("filename")
        .reset_index(drop=True)
    )

    print(
        f"\nRelevant notebooks found: "
        f"{len(notebook_df):,}"
    )

    display(notebook_df)

else:

    print("\nNo COST notebooks found.")


# ============================================================
# 11. SAFETY STATUS
# ============================================================

print("\n" + "=" * 100)
print("CELL 1 STATUS")
print("=" * 100)

print(
    f"""
✓ Frozen COST Improved checkpoint located
✓ Checkpoint loaded successfully
✓ Architecture dimensions verified: 93 temporal + 11 static
✓ Frozen MIMIC-IV normalization artifacts verified
✓ Existing reviewer-related artifacts inventoried
✓ Existing COST notebooks inventoried

NO TRAINING PERFORMED
NO FINE-TUNING PERFORMED
NO MODEL INFERENCE PERFORMED
NO EXTERNAL VALIDATION RE-RUN
NO DATA MODIFIED

NEXT:
CELL 2 — Reviewer 2 Concern 3
         Controlled-sparsity distribution audit
         (patient-level + latent-feature-level)
"""
)

print("=" * 100)

In [ ]:
# ============================================================
# CELL 2 — REVIEWER 2: SPARSITY ARTIFACT AUDIT
#
# PURPOSE:
#   Inspect the EXISTING controlled-sparsity results.
#
# IMPORTANT:
#   NO TRAINING
#   NO INFERENCE
#   NO RECOMPUTATION
#   NO MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import torch

print("=" * 100)
print("REVIEWER 2 — CONTROLLED-SPARSITY ARTIFACT AUDIT")
print("=" * 100)

REV_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/"
    "mimic-iv-2.1/revision2_outputs"
)

PER_SEED = REV_DIR / "table_xxv_sparsity_per_seed.csv"
SUMMARY  = REV_DIR / "table_xxv_sparsity_summary.csv"

assert PER_SEED.exists(), f"Not found: {PER_SEED}"
assert SUMMARY.exists(),  f"Not found: {SUMMARY}"


# ============================================================
# 1. LOAD EXISTING SPARSITY TABLES
# ============================================================

per_seed = pd.read_csv(PER_SEED)
summary  = pd.read_csv(SUMMARY)

print("\n" + "=" * 100)
print("EXISTING SPARSITY RESULTS — PER SEED")
print("=" * 100)

print("\nShape:", per_seed.shape)
print("Columns:", list(per_seed.columns))
print()

display(per_seed)


print("\n" + "=" * 100)
print("EXISTING SPARSITY RESULTS — SUMMARY")
print("=" * 100)

print("\nShape:", summary.shape)
print("Columns:", list(summary.columns))
print()

display(summary)


# ============================================================
# 2. CHECK GATING ON/OFF ARTIFACT COMPLETENESS
# ============================================================

print("\n" + "=" * 100)
print("GATING EXPERIMENT COMPLETENESS")
print("=" * 100)

rows = []

for mode in ["ON", "OFF"]:
    for seed in [42, 43, 44]:

        best = REV_DIR / f"gating_{mode}_seed{seed}_best.pt"
        hist = REV_DIR / f"gating_{mode}_seed{seed}_history.csv"
        pred = REV_DIR / f"gating_{mode}_seed{seed}_predictions.npz"
        snap = REV_DIR / f"gating_{mode}_seed{seed}_snapshots.pt"

        rows.append({
            "gating": mode,
            "seed": seed,
            "best_checkpoint": best.exists(),
            "history": hist.exists(),
            "predictions": pred.exists(),
            "snapshots": snap.exists()
        })

artifact_check = pd.DataFrame(rows)

display(artifact_check)

assert artifact_check[
    [
        "best_checkpoint",
        "history",
        "predictions",
        "snapshots"
    ]
].all().all()

print("\n✓ All gating ON/OFF artifacts are present")


# ============================================================
# 3. INSPECT HISTORY FILE STRUCTURE
# ============================================================

print("\n" + "=" * 100)
print("GATING TRAINING-HISTORY STRUCTURE")
print("=" * 100)

for mode in ["ON", "OFF"]:
    for seed in [42, 43, 44]:

        f = REV_DIR / f"gating_{mode}_seed{seed}_history.csv"

        h = pd.read_csv(f)

        print(
            f"\n{f.name}"
            f"\n  shape   : {h.shape}"
            f"\n  columns : {list(h.columns)}"
        )

        print(h.tail(3).to_string(index=False))


# ============================================================
# 4. INSPECT PREDICTION FILE STRUCTURE
# ============================================================

print("\n" + "=" * 100)
print("GATING PREDICTION ARTIFACT STRUCTURE")
print("=" * 100)

for mode in ["ON", "OFF"]:
    for seed in [42, 43, 44]:

        f = REV_DIR / f"gating_{mode}_seed{seed}_predictions.npz"

        z = np.load(f, allow_pickle=True)

        print(f"\n{f.name}")
        print("  keys:", z.files)

        for key in z.files:

            arr = np.asarray(z[key])

            if arr.size > 0 and np.issubdtype(arr.dtype, np.number):
                print(
                    f"  {key:25s} "
                    f"shape={arr.shape} "
                    f"dtype={arr.dtype} "
                    f"min={np.nanmin(arr):.6f} "
                    f"max={np.nanmax(arr):.6f}"
                )
            else:
                print(
                    f"  {key:25s} "
                    f"shape={arr.shape} "
                    f"dtype={arr.dtype}"
                )


# ============================================================
# 5. INSPECT CHECKPOINT CONTENT
# ============================================================

print("\n" + "=" * 100)
print("GATING CHECKPOINT CONTENT")
print("=" * 100)

for mode in ["ON", "OFF"]:
    for seed in [42, 43, 44]:

        f = REV_DIR / f"gating_{mode}_seed{seed}_best.pt"

        obj = torch.load(
            f,
            map_location="cpu",
            weights_only=False
        )

        print(f"\n{f.name}")

        if isinstance(obj, dict):

            print("  keys:", list(obj.keys()))

            for key, value in obj.items():

                if isinstance(
                    value,
                    (int, float, np.integer, np.floating)
                ):
                    print(
                        f"    {key:25s}: {value}"
                    )

                elif torch.is_tensor(value) and value.numel() == 1:
                    print(
                        f"    {key:25s}: {value.item()}"
                    )

        else:
            print(
                "  object type:",
                type(obj).__name__
            )


# ============================================================
# 6. BASIC CONSISTENCY CHECK OF EXISTING SPARSITY TABLE
# ============================================================

print("\n" + "=" * 100)
print("SPARSITY TABLE CONSISTENCY")
print("=" * 100)

print("\nPER-SEED TABLE:")
print(per_seed.to_string(index=False))

print("\nSUMMARY TABLE:")
print(summary.to_string(index=False))


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("CELL 2 COMPLETE")
print("=" * 100)

print("""
✓ Existing sparsity tables loaded
✓ Gating ON/OFF artifacts verified
✓ Seeds 42 / 43 / 44 verified
✓ History structures inspected
✓ Prediction structures inspected
✓ Checkpoint metadata inspected

NO TRAINING
NO INFERENCE
NO RESULTS MODIFIED

NEXT:
Use these outputs to determine whether Reviewer 2's
requested sparsity-distribution evidence is already sufficient
or whether ONE targeted extraction is still required.
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 3 — REVIEWER 2 COMPUTATIONAL-EFFICIENCY ARTIFACT AUDIT
#
# PURPOSE:
#   Determine what computational-efficiency evidence already
#   exists before calculating anything new.
#
# NO TRAINING
# NO MODEL INFERENCE
# NO MODIFICATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")
REV_DIR = ROOT / "mimic-iv-2.1" / "revision2_outputs"

print("=" * 100)
print("REVIEWER 2 — COMPUTATIONAL-EFFICIENCY ARTIFACT AUDIT")
print("=" * 100)


# ============================================================
# 1. SEARCH FILE NAMES
# ============================================================

keywords = [
    "flop",
    "flops",
    "latency",
    "runtime",
    "memory",
    "parameter",
    "params",
    "complexity",
    "efficiency",
    "computational",
    "inference",
    "throughput"
]

hits = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    matched = [k for k in keywords if k in name]

    if matched:
        hits.append({
            "filename": p.name,
            "matched": ", ".join(matched),
            "size_mb": p.stat().st_size / (1024**2),
            "path": str(p)
        })

file_hits = pd.DataFrame(hits)

print("\n" + "=" * 100)
print("FILES WITH COMPUTATIONAL-EFFICIENCY KEYWORDS")
print("=" * 100)

if len(file_hits):
    display(file_hits.sort_values("filename").reset_index(drop=True))
else:
    print("No filename-level computational artifacts found.")


# ============================================================
# 2. SEARCH NOTEBOOK SOURCE TEXT
# ============================================================

print("\n" + "=" * 100)
print("SEARCHING NOTEBOOKS FOR COMPUTATIONAL METRICS")
print("=" * 100)

notebook_hits = []

patterns = [
    r"\bflops?\b",
    r"\blatency\b",
    r"\binference time\b",
    r"\bruntime\b",
    r"\bthroughput\b",
    r"\bpeak memory\b",
    r"\bmemory usage\b",
    r"\bparameters\b",
    r"\bparam count\b",
    r"\bcomputational complexity\b"
]

for p in ROOT.rglob("*.ipynb"):

    try:
        text = p.read_text(
            encoding="utf-8",
            errors="ignore"
        ).lower()

        found = []

        for pattern in patterns:
            if re.search(pattern, text):
                found.append(pattern)

        if found:
            notebook_hits.append({
                "notebook": p.name,
                "matches": ", ".join(found),
                "path": str(p)
            })

    except Exception:
        pass


nb_df = pd.DataFrame(notebook_hits)

if len(nb_df):
    display(
        nb_df.sort_values("notebook")
             .reset_index(drop=True)
    )
else:
    print("No notebook text matches found.")


# ============================================================
# 3. EXACT PARAMETER COUNT FROM FROZEN CHECKPOINT
# ============================================================

print("\n" + "=" * 100)
print("FROZEN COST IMPROVED PARAMETER COUNT")
print("=" * 100)

state = ckpt["model_state_dict"]

# Count trainable tensors represented by state dictionary,
# excluding registered non-parameter positional buffer.
parameter_count = 0
buffer_count = 0

for name, tensor in state.items():

    n = tensor.numel()

    if name == "pos_enc.pe":
        buffer_count += n
    else:
        parameter_count += n

print(f"Trainable parameter count : {parameter_count:,}")
print(f"Registered buffer count   : {buffer_count:,}")
print(f"Stored tensor elements    : {parameter_count + buffer_count:,}")

# expected from strict architecture load
assert parameter_count == 3_653_890

print("\n✓ Exact COST Improved parameter count verified")


# ============================================================
# 4. MODEL SIZE ESTIMATE
# ============================================================

bytes_fp32 = parameter_count * 4
model_mb_fp32 = bytes_fp32 / (1024**2)

print("\n" + "=" * 100)
print("MODEL PARAMETER STORAGE")
print("=" * 100)

print(f"FP32 parameter storage : {model_mb_fp32:.2f} MB")


# ============================================================
# 5. CHECK WHETHER THOP / FVCORE ARE AVAILABLE
#    IMPORTANT: DO NOT CALCULATE YET
# ============================================================

print("\n" + "=" * 100)
print("FLOP TOOL AVAILABILITY")
print("=" * 100)

try:
    import thop
    print("THOP   : AVAILABLE")
except Exception:
    print("THOP   : NOT INSTALLED")

try:
    import fvcore
    print("fvcore : AVAILABLE")
except Exception:
    print("fvcore : NOT INSTALLED")


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("CELL 3 COMPLETE")
print("=" * 100)

print("""
✓ Existing computational artifacts searched
✓ Existing notebooks searched
✓ Exact parameter count verified
✓ FP32 parameter storage calculated
✓ FLOP-tool availability checked

NO TRAINING
NO INFERENCE
NO MODEL MODIFICATION

Do NOT install anything yet.
Do NOT calculate FLOPs yet.

The next step depends on whether the original experiments
already contain latency/FLOPs/memory measurements.
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 4 — REVIEWER 2 COMPUTATIONAL-EFFICIENCY RESULTS
#
# Read EXISTING revision artifacts only.
#
# NO TRAINING
# NO INFERENCE
# NO PROFILING
# NO RECOMPUTATION
# ============================================================

from pathlib import Path
import pandas as pd

REV_DIR = Path(
    "/content/drive/MyDrive/COST_PROJECT/"
    "mimic-iv-2.1/revision2_outputs"
)

DETAIL_FILE = REV_DIR / "table_xxvi_computational_detailed.csv"
SUMMARY_FILE = REV_DIR / "table_xxvi_computational_summary.csv"

print("=" * 100)
print("REVIEWER 2 — EXISTING COMPUTATIONAL-EFFICIENCY RESULTS")
print("=" * 100)

assert DETAIL_FILE.exists(), f"Missing: {DETAIL_FILE}"
assert SUMMARY_FILE.exists(), f"Missing: {SUMMARY_FILE}"

# ============================================================
# 1. LOAD TABLES
# ============================================================

detail = pd.read_csv(DETAIL_FILE)
summary = pd.read_csv(SUMMARY_FILE)

print("\n" + "=" * 100)
print("DETAILED COMPUTATIONAL RESULTS")
print("=" * 100)

print("Shape   :", detail.shape)
print("Columns :", list(detail.columns))
print()

display(detail)

print("\nRAW:")
print(detail.to_string(index=False))


print("\n" + "=" * 100)
print("SUMMARY COMPUTATIONAL RESULTS")
print("=" * 100)

print("Shape   :", summary.shape)
print("Columns :", list(summary.columns))
print()

display(summary)

print("\nRAW:")
print(summary.to_string(index=False))


# ============================================================
# 2. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 100)
print("MISSING-VALUE AUDIT")
print("=" * 100)

print("\nDetailed:")
print(detail.isna().sum().to_string())

print("\nSummary:")
print(summary.isna().sum().to_string())


# ============================================================
# 3. REPORT NUMERIC RANGES
# ============================================================

print("\n" + "=" * 100)
print("NUMERIC RANGE AUDIT")
print("=" * 100)

for name, df in [
    ("DETAILED", detail),
    ("SUMMARY", summary)
]:
    print(f"\n{name}")

    for col in df.select_dtypes(include="number").columns:
        x = df[col].dropna()

        if len(x):
            print(
                f"{col:30s} "
                f"min={x.min():.6f} | "
                f"max={x.max():.6f}"
            )


# ============================================================
# 4. SEARCH ORIGINAL FIGURE
# ============================================================

FIG = (
    Path("/content/drive/MyDrive/COST_PROJECT/")
    / "mimic-iv-2.1"
    / "result 23 April 2026 10:42"
    / "efficiency_performance.png"
)

print("\n" + "=" * 100)
print("EXISTING EFFICIENCY FIGURE")
print("=" * 100)

print("Path   :", FIG)
print("Exists :", FIG.exists())

if FIG.exists():
    print(
        "Size   :",
        round(FIG.stat().st_size / 1024, 2),
        "KB"
    )


# ============================================================
# 5. CROSS-CHECK COST PARAMETER COUNT
# ============================================================

print("\n" + "=" * 100)
print("COST IMPROVED PARAMETER REFERENCE")
print("=" * 100)

print("Verified trainable parameters : 3,653,890")
print("FP32 parameter storage        : 13.94 MB")


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 4 COMPLETE")
print("=" * 100)

print("""
✓ Existing detailed computational table loaded
✓ Existing summary computational table loaded
✓ Missing values audited
✓ Numeric ranges audited
✓ Existing efficiency figure located
✓ COST parameter count cross-reference available

NO TRAINING
NO INFERENCE
NO NEW PROFILING
NO RESULTS MODIFIED
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 5 — REVIEWER 2 STATISTICAL SIGNIFICANCE ARTIFACT AUDIT
#
# PURPOSE:
#   Read existing significance-testing artifacts only.
#
# NO TRAINING
# NO INFERENCE
# NO RECOMPUTATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")
REV_DIR = ROOT / "mimic-iv-2.1" / "revision2_outputs"

print("=" * 100)
print("REVIEWER 2 — STATISTICAL SIGNIFICANCE ARTIFACT AUDIT")
print("=" * 100)


# ============================================================
# 1. FIND RELEVANT FILES
# ============================================================

keywords = [
    "significance",
    "delong",
    "bootstrap",
    "statistical",
    "comparison",
    "pairwise",
    "confidence",
    "ci"
]

hits = []

for p in REV_DIR.rglob("*"):

    if not p.is_file():
        continue

    name = p.name.lower()

    matched = [k for k in keywords if k in name]

    if matched:
        hits.append({
            "filename": p.name,
            "matched": ", ".join(matched),
            "size_kb": p.stat().st_size / 1024,
            "path": str(p)
        })

hits_df = pd.DataFrame(hits)

print("\n" + "=" * 100)
print("CANDIDATE STATISTICAL FILES")
print("=" * 100)

if len(hits_df):
    display(
        hits_df.sort_values("filename")
               .reset_index(drop=True)
    )
else:
    print("No filename-level statistical artifacts found.")


# ============================================================
# 2. READ ALL SMALL CSV FILES MATCHING THE SEARCH
# ============================================================

print("\n" + "=" * 100)
print("EXISTING STATISTICAL TABLE CONTENT")
print("=" * 100)

csv_files = []

if len(hits_df):

    for path in hits_df["path"]:

        p = Path(path)

        if p.suffix.lower() == ".csv":
            csv_files.append(p)

csv_files = list(dict.fromkeys(csv_files))

if not csv_files:

    print("No matching CSV files found.")

else:

    for p in csv_files:

        print("\n" + "-" * 100)
        print("FILE:", p.name)
        print("-" * 100)

        try:
            df = pd.read_csv(p)

            print("Shape   :", df.shape)
            print("Columns :", list(df.columns))
            print()

            display(df)

            print("\nRAW:")
            print(df.to_string(index=False))

            print("\nMissing values:")
            print(df.isna().sum().to_string())

        except Exception as e:

            print(
                "Could not read as standard CSV:",
                repr(e)
            )


# ============================================================
# 3. SEARCH ALL REVISION TABLES FOR P-VALUE / CI / DELONG
# ============================================================

print("\n" + "=" * 100)
print("SEARCHING ALL REVISION CSV TABLES FOR STATISTICAL COLUMNS")
print("=" * 100)

stat_column_hits = []

for p in REV_DIR.glob("*.csv"):

    try:

        df = pd.read_csv(p)

        columns = [str(c) for c in df.columns]

        matching_columns = [
            c for c in columns
            if any(
                term in c.lower()
                for term in [
                    "p-value",
                    "p_value",
                    "p value",
                    "delong",
                    "confidence",
                    "95% ci",
                    "ci",
                    "bootstrap",
                    "difference"
                ]
            )
        ]

        if matching_columns:

            stat_column_hits.append({
                "filename": p.name,
                "shape": str(df.shape),
                "statistical_columns":
                    ", ".join(matching_columns),
                "path": str(p)
            })

    except Exception:
        pass


stat_cols_df = pd.DataFrame(stat_column_hits)

if len(stat_cols_df):

    display(
        stat_cols_df
        .sort_values("filename")
        .reset_index(drop=True)
    )

else:

    print(
        "No statistical columns found "
        "in revision CSV tables."
    )


# ============================================================
# 4. FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("CELL 5 COMPLETE")
print("=" * 100)

print("""
✓ Existing statistical artifacts searched
✓ Existing significance tables inspected
✓ P-value / CI / DeLong-related columns searched

NO TRAINING
NO INFERENCE
NO BOOTSTRAP RE-RUN
NO DELONG RE-RUN
NO RESULTS MODIFIED

NEXT:
Determine whether the existing statistical evidence
already answers the reviewer or whether one targeted
calculation is still genuinely required.
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 5B — REVIEWER 2 CONCERN 5
# PREDICTION TRACEABILITY DISCOVERY
#
# PURPOSE:
# Find ALL existing prediction artifacts that could have been
# used for Table XXVI / DeLong / bootstrap comparisons.
#
# NO TRAINING
# NO INFERENCE
# NO STATISTICAL RECOMPUTATION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

print("=" * 100)
print("REVIEWER 2 CONCERN 5 — PREDICTION TRACEABILITY DISCOVERY")
print("=" * 100)

# ------------------------------------------------------------
# 1. Find prediction-like artifacts
# ------------------------------------------------------------

keywords = [
    "prediction",
    "predictions",
    "pred",
    "prob",
    "probability",
    "delong",
    "bootstrap",
    "significance"
]

extensions = {
    ".npz", ".npy", ".csv",
    ".pkl", ".pickle",
    ".pt", ".pth"
}

rows = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in extensions:
        continue

    name = p.name.lower()

    matched = [k for k in keywords if k in name]

    if matched:

        rows.append({
            "filename": p.name,
            "extension": p.suffix.lower(),
            "matched": ", ".join(matched),
            "size_kb": p.stat().st_size / 1024,
            "path": str(p)
        })

pred_files = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("PREDICTION-LIKE ARTIFACTS")
print("=" * 100)

if len(pred_files):

    pred_files = (
        pred_files
        .sort_values(["filename", "path"])
        .reset_index(drop=True)
    )

    display(pred_files)

else:

    print("No prediction-like artifacts found.")


# ------------------------------------------------------------
# 2. Inspect NPZ structures
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("NPZ CONTENT AUDIT")
print("=" * 100)

if len(pred_files):

    npz_files = pred_files[
        pred_files["extension"] == ".npz"
    ]

    for _, row in npz_files.iterrows():

        p = Path(row["path"])

        print("\n" + "-" * 100)
        print("FILE:", p)
        print("-" * 100)

        try:

            z = np.load(p, allow_pickle=True)

            print("Keys:", z.files)

            for key in z.files:

                arr = np.asarray(z[key])

                msg = (
                    f"  {key:25s} "
                    f"shape={arr.shape} "
                    f"dtype={arr.dtype}"
                )

                if (
                    arr.size > 0 and
                    np.issubdtype(arr.dtype, np.number)
                ):

                    finite = arr[np.isfinite(arr)]

                    if finite.size:

                        msg += (
                            f" min={finite.min():.6f}"
                            f" max={finite.max():.6f}"
                            f" mean={finite.mean():.6f}"
                        )

                print(msg)

        except Exception as e:

            print("ERROR:", repr(e))


# ------------------------------------------------------------
# 3. Search notebooks for the suspicious AUC = 0.6496
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCHING NOTEBOOKS FOR AUC 0.6496")
print("=" * 100)

auc_hits = []

for p in ROOT.rglob("*.ipynb"):

    try:

        txt = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        if (
            "0.6496" in txt or
            "6496" in txt
        ):

            auc_hits.append({
                "notebook": p.name,
                "path": str(p)
            })

    except Exception:
        pass

auc_df = pd.DataFrame(auc_hits)

if len(auc_df):

    display(auc_df)

else:

    print("No notebook containing 0.6496 found.")


# ------------------------------------------------------------
# 4. Search CSV files for 0.6496
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("SEARCHING CSV FILES FOR AUC 0.6496")
print("=" * 100)

csv_hits = []

for p in ROOT.rglob("*.csv"):

    try:

        txt = p.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        if (
            "0.6496" in txt or
            "6496" in txt
        ):

            csv_hits.append({
                "filename": p.name,
                "path": str(p)
            })

    except Exception:
        pass

csv_auc_df = pd.DataFrame(csv_hits)

if len(csv_auc_df):

    display(csv_auc_df)

else:

    print("No CSV containing 0.6496 found.")


# ------------------------------------------------------------
# 5. Search notebooks for DeLong/bootstrap code
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOKS CONTAINING DELONG / BOOTSTRAP")
print("=" * 100)

stat_hits = []

for p in ROOT.rglob("*.ipynb"):

    try:

        txt = p.read_text(
            encoding="utf-8",
            errors="ignore"
        ).lower()

        found = []

        if "delong" in txt:
            found.append("DeLong")

        if "bootstrap" in txt:
            found.append("bootstrap")

        if found:

            stat_hits.append({
                "notebook": p.name,
                "methods": ", ".join(found),
                "path": str(p)
            })

    except Exception:
        pass

stat_nb_df = pd.DataFrame(stat_hits)

if len(stat_nb_df):

    display(
        stat_nb_df
        .sort_values("notebook")
        .reset_index(drop=True)
    )

else:

    print("No DeLong/bootstrap notebook found.")


# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("CELL 5B COMPLETE")
print("=" * 100)

print("""
✓ Prediction artifacts inventoried
✓ NPZ contents inspected
✓ Suspicious representative AUC 0.6496 searched
✓ Existing DeLong/bootstrap notebooks located

NO TRAINING
NO INFERENCE
NO STATISTICAL TEST RE-RUN

NEXT:
Identify EXACT run/prediction files corresponding to
Table XXVI before performing any statistical calculation.
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 5C — REVIEWER 2 CONCERN 5
# EXACT STATISTICAL TRACEABILITY AUDIT
#
# PURPOSE:
# 1. Verify AUROC directly from saved prediction files
# 2. Confirm all compared files use identical test labels
# 3. Read existing significance tables
# 4. Identify provenance clues for Initial COST AUC = 0.6496
#
# NO TRAINING
# NO MODEL INFERENCE
# NO NEW DELONG TEST
# NO NEW BOOTSTRAP
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
import json
import re

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")
PRED_DIR = ROOT / "mimic-iv-2.1" / "saved_predictions"

print("=" * 100)
print("REVIEWER 2 CONCERN 5 — EXACT STATISTICAL TRACEABILITY")
print("=" * 100)


# ============================================================
# 1. LOAD ALL RELEVANT SAVED PREDICTIONS
# ============================================================

prediction_files = {
    "Initial COST":
        PRED_DIR / "cost_before_predictions.npz",

    "Improved COST":
        PRED_DIR / "cost_improved_predictions.npz",

    "Baseline LSTM":
        PRED_DIR / "baseline_lstm_predictions.npz",

    "Attention LSTM":
        PRED_DIR / "baseline_attn_lstm_predictions.npz",

    "TCN":
        PRED_DIR / "baseline_tcn_predictions.npz",

    "Logistic Regression":
        PRED_DIR / "baseline_lr_predictions.npz",

    "Random Forest":
        PRED_DIR / "baseline_rf_predictions.npz",
}

for name, path in prediction_files.items():
    assert path.exists(), f"Missing {name}: {path}"

print("\nAll required prediction files found.")


# ============================================================
# 2. DIRECTLY RECOMPUTE DESCRIPTIVE AUROC/AUPRC
#
# NOTE:
# This is NOT a new experiment.
# It simply verifies the metrics encoded in existing
# saved probability files.
# ============================================================

print("\n" + "=" * 100)
print("DIRECT METRIC VERIFICATION FROM SAVED PROBABILITIES")
print("=" * 100)

results = []
labels_by_model = {}

for name, path in prediction_files.items():

    z = np.load(path, allow_pickle=True)

    if "test_labels" not in z.files:
        raise KeyError(
            f"{name}: test_labels not found. Keys={z.files}"
        )

    y = np.asarray(z["test_labels"]).astype(int)
    labels_by_model[name] = y

    # Improved COST contains single + ensemble
    if name == "Improved COST":

        for pred_key, label in [
            ("probs_single", "Improved COST — Single"),
            ("probs_ensemble", "Improved COST — Ensemble")
        ]:

            p = np.asarray(z[pred_key]).astype(float)

            auc = roc_auc_score(y, p)
            auprc = average_precision_score(y, p)

            results.append({
                "Model": label,
                "Prediction file": path.name,
                "Prediction key": pred_key,
                "N": len(y),
                "Deaths": int(y.sum()),
                "AUROC": auc,
                "AUPRC": auprc
            })

    else:

        if "probs" not in z.files:
            raise KeyError(
                f"{name}: probs not found. Keys={z.files}"
            )

        p = np.asarray(z["probs"]).astype(float)

        auc = roc_auc_score(y, p)
        auprc = average_precision_score(y, p)

        results.append({
            "Model": name,
            "Prediction file": path.name,
            "Prediction key": "probs",
            "N": len(y),
            "Deaths": int(y.sum()),
            "AUROC": auc,
            "AUPRC": auprc
        })


metric_df = pd.DataFrame(results)

display(
    metric_df.style.format({
        "AUROC": "{:.6f}",
        "AUPRC": "{:.6f}"
    })
)

print("\nRAW:")
print(
    metric_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}"
    )
)


# ============================================================
# 3. CRITICAL CHECK — DOES INITIAL COST REALLY EQUAL 0.6496?
# ============================================================

initial_row = metric_df[
    metric_df["Model"] == "Initial COST"
].iloc[0]

initial_auc = float(initial_row["AUROC"])

print("\n" + "=" * 100)
print("INITIAL COST 0.6496 AUDIT")
print("=" * 100)

print(f"Saved prediction file : cost_before_predictions.npz")
print(f"Direct AUROC          : {initial_auc:.10f}")
print(f"Rounded to 4 decimals : {initial_auc:.4f}")
print(f"Paper footnote value  : 0.6496")

if round(initial_auc, 4) == 0.6496:

    print("\n✓ CONFIRMED:")
    print(
        "  The reported 0.6496 comes directly from "
        "cost_before_predictions.npz."
    )

else:

    print("\n⚠ NOT CONFIRMED:")
    print(
        "  cost_before_predictions.npz does NOT reproduce "
        "the reported 0.6496."
    )


# ============================================================
# 4. TEST-COHORT LABEL ALIGNMENT
# ============================================================

print("\n" + "=" * 100)
print("TEST-COHORT LABEL ALIGNMENT")
print("=" * 100)

reference_name = "Initial COST"
reference_y = labels_by_model[reference_name]

alignment_rows = []

for name, y in labels_by_model.items():

    same_shape = y.shape == reference_y.shape

    identical = (
        same_shape and
        np.array_equal(y, reference_y)
    )

    alignment_rows.append({
        "Model": name,
        "N": len(y),
        "Deaths": int(y.sum()),
        "Same shape as Initial COST": same_shape,
        "Labels identical to Initial COST": identical
    })

alignment_df = pd.DataFrame(alignment_rows)

display(alignment_df)

if alignment_df[
    "Labels identical to Initial COST"
].all():

    print(
        "\n✓ All saved prediction files use "
        "identical test labels/order."
    )

else:

    print(
        "\n⚠ WARNING: At least one prediction file "
        "does not use identical labels/order."
    )


# ============================================================
# 5. READ EXISTING SIGNIFICANCE TABLES
# ============================================================

print("\n" + "=" * 100)
print("EXISTING SIGNIFICANCE TABLES")
print("=" * 100)

significance_files = [
    PRED_DIR / "table_B_significance_testing.csv",
    PRED_DIR / "table_XXIV_significance_testing.csv",
]

for path in significance_files:

    print("\n" + "-" * 100)
    print("FILE:", path.name)
    print("-" * 100)

    if not path.exists():

        print("NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Shape   :", df.shape)
    print("Columns :", list(df.columns))
    print()

    display(df)

    print("\nRAW:")
    print(df.to_string(index=False))


# ============================================================
# 6. SEARCH NOTEBOOK SOURCE FOR EXACT 0.6496 CONTEXT
# ============================================================

print("\n" + "=" * 100)
print("0.6496 NOTEBOOK PROVENANCE CONTEXT")
print("=" * 100)

candidate_notebooks = [
    ROOT / "COST_Master_AllExperiments.ipynb",
    ROOT / "COST_Master_AllExperiments_CLEAN.ipynb",
    ROOT / "COST_RevisionII_FINAL_SelfContained (1).ipynb",
    ROOT / "COST_Improved_eICU_FULL_External_Validation.ipynb",
]

context_rows = []

for nb_path in candidate_notebooks:

    if not nb_path.exists():
        continue

    try:
        nb = json.loads(
            nb_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

        for cell_idx, cell in enumerate(nb.get("cells", [])):

            # Search both code/source and stored outputs
            source = "".join(cell.get("source", []))

            output_text_parts = []

            for out in cell.get("outputs", []):

                if "text" in out:
                    output_text_parts.extend(out["text"])

                if "data" in out:

                    data = out["data"]

                    if "text/plain" in data:
                        output_text_parts.extend(
                            data["text/plain"]
                        )

            output_text = "".join(output_text_parts)

            combined = source + "\n" + output_text

            if "0.6496" in combined:

                # keep compact context around occurrence
                pos = combined.find("0.6496")

                start = max(0, pos - 500)
                end = min(len(combined), pos + 700)

                context = combined[start:end]

                context_rows.append({
                    "Notebook": nb_path.name,
                    "Cell index": cell_idx,
                    "Context": context
                })

    except Exception as e:

        print(
            f"Could not inspect {nb_path.name}:",
            repr(e)
        )


if context_rows:

    context_df = pd.DataFrame(context_rows)

    for _, row in context_df.iterrows():

        print("\n" + "-" * 100)
        print(
            f"NOTEBOOK : {row['Notebook']}\n"
            f"CELL     : {row['Cell index']}"
        )
        print("-" * 100)
        print(row["Context"])

else:

    print("No readable 0.6496 context found.")


# ============================================================
# 7. SEARCH FOR COST-BEFORE FILE USAGE
# ============================================================

print("\n" + "=" * 100)
print("SEARCHING NOTEBOOKS FOR cost_before_predictions.npz USAGE")
print("=" * 100)

usage_rows = []

for nb_path in ROOT.glob("*.ipynb"):

    try:

        nb = json.loads(
            nb_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

        for cell_idx, cell in enumerate(nb.get("cells", [])):

            source = "".join(cell.get("source", []))

            if "cost_before_predictions" in source:

                usage_rows.append({
                    "Notebook": nb_path.name,
                    "Cell index": cell_idx,
                    "Source": source[:2500]
                })

    except Exception:
        pass


if usage_rows:

    usage_df = pd.DataFrame(usage_rows)

    for _, row in usage_df.iterrows():

        print("\n" + "-" * 100)
        print(
            f"NOTEBOOK : {row['Notebook']}\n"
            f"CELL     : {row['Cell index']}"
        )
        print("-" * 100)
        print(row["Source"])

else:

    print(
        "No notebook source explicitly referencing "
        "cost_before_predictions.npz found."
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 5C COMPLETE")
print("=" * 100)

print("""
✓ Existing saved probabilities directly verified
✓ Initial COST 0.6496 provenance tested
✓ Test-label alignment checked
✓ Existing significance tables opened
✓ Notebook context for 0.6496 inspected
✓ cost_before_predictions.npz usage traced

NO TRAINING
NO MODEL INFERENCE
NO NEW DELONG TEST
NO NEW BOOTSTRAP
NO RESULTS MODIFIED

NEXT:
Use this audit to decide whether Reviewer 2 Concern #5
can be resolved by corrected traceability alone or whether
between-run statistical analysis must be added.
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 5D — REVIEWER 2 CONCERN 5
# REPEATED-RUN PREDICTION ARTIFACT DISCOVERY
#
# PURPOSE:
# Find whether patient-level predictions from individual runs
# still exist for:
#   - Initial COST
#   - Improved COST
#   - Baseline LSTM
#   - Attention LSTM
#
# NO TRAINING
# NO MODEL INFERENCE
# NO DELONG
# NO BOOTSTRAP
# NO RESULTS MODIFIED
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

print("=" * 100)
print("REVIEWER 2 CONCERN 5 — REPEATED-RUN ARTIFACT DISCOVERY")
print("=" * 100)


# ============================================================
# 1. SEARCH FILESYSTEM FOR RUN/SEED PREDICTION FILES
# ============================================================

extensions = {
    ".npz", ".npy", ".csv",
    ".pkl", ".pickle",
    ".pt", ".pth"
}

prediction_terms = [
    "pred",
    "prediction",
    "prob",
    "probability"
]

run_terms = [
    "run",
    "seed",
    "fold",
    "repeat"
]

model_terms = [
    "cost",
    "dsrnn",
    "lstm",
    "attention",
    "attn"
]

rows = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in extensions:
        continue

    text = str(p).lower()

    has_prediction = any(
        term in text
        for term in prediction_terms
    )

    has_run = any(
        term in text
        for term in run_terms
    )

    has_model = any(
        term in text
        for term in model_terms
    )

    if has_prediction and has_model:

        rows.append({
            "filename": p.name,
            "extension": p.suffix.lower(),
            "run_or_seed_in_path": has_run,
            "size_kb": p.stat().st_size / 1024,
            "path": str(p)
        })


artifact_df = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("PREDICTION-LIKE MODEL ARTIFACTS")
print("=" * 100)

if len(artifact_df):

    artifact_df = (
        artifact_df
        .sort_values(
            ["run_or_seed_in_path", "filename"],
            ascending=[False, True]
        )
        .reset_index(drop=True)
    )

    display(artifact_df)

else:

    print("No candidate artifacts found.")


# ============================================================
# 2. SHOW ONLY FILES EXPLICITLY CONTAINING RUN / SEED
# ============================================================

print("\n" + "=" * 100)
print("EXPLICIT RUN / SEED PREDICTION CANDIDATES")
print("=" * 100)

if len(artifact_df):

    run_df = artifact_df[
        artifact_df["run_or_seed_in_path"]
    ].copy()

    if len(run_df):
        display(run_df.reset_index(drop=True))
    else:
        print(
            "No prediction filenames/paths explicitly "
            "contain run/seed/fold/repeat."
        )

else:
    print("No artifacts available.")


# ============================================================
# 3. INSPECT NPZ CANDIDATES
# ============================================================

print("\n" + "=" * 100)
print("NPZ STRUCTURE AUDIT")
print("=" * 100)

if len(artifact_df):

    npz_df = artifact_df[
        artifact_df["extension"] == ".npz"
    ]

    for _, row in npz_df.iterrows():

        p = Path(row["path"])

        print("\n" + "-" * 100)
        print("FILE:", p)
        print("-" * 100)

        try:

            z = np.load(p, allow_pickle=True)

            print("Keys:", z.files)

            for key in z.files:

                arr = np.asarray(z[key])

                print(
                    f"  {key:30s} "
                    f"shape={str(arr.shape):18s} "
                    f"dtype={arr.dtype}"
                )

        except Exception as e:

            print("Could not inspect:", repr(e))


# ============================================================
# 4. SEARCH NOTEBOOK CODE FOR SAVING PER-RUN PREDICTIONS
# ============================================================

print("\n" + "=" * 100)
print("SEARCHING NOTEBOOKS FOR PER-RUN PREDICTION SAVING")
print("=" * 100)

patterns = [
    r"np\.savez",
    r"np\.save",
    r"predictions",
    r"test_probs",
    r"probs",
    r"best_run",
    r"all_results",
]

notebook_hits = []

for nb_path in ROOT.rglob("*.ipynb"):

    try:

        nb = json.loads(
            nb_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

        for cell_idx, cell in enumerate(
            nb.get("cells", [])
        ):

            source = "".join(
                cell.get("source", [])
            )

            source_lower = source.lower()

            # Require prediction-related code
            if not any(
                x in source_lower
                for x in [
                    "test_probs",
                    "prediction",
                    "np.save",
                    "np.savez",
                    "best_run",
                    "all_results"
                ]
            ):
                continue

            # Look specifically for run loops
            run_context = any(
                x in source_lower
                for x in [
                    "for run",
                    "for seed",
                    "runs =",
                    "all_results",
                    "best_run"
                ]
            )

            if run_context:

                notebook_hits.append({
                    "notebook": nb_path.name,
                    "cell": cell_idx,
                    "source": source[:4000]
                })

    except Exception:
        pass


if notebook_hits:

    nb_df = pd.DataFrame(notebook_hits)

    print(
        f"Candidate notebook cells: "
        f"{len(nb_df)}"
    )

    for _, row in nb_df.iterrows():

        print("\n" + "-" * 100)
        print(
            "NOTEBOOK:",
            row["notebook"]
        )
        print(
            "CELL    :",
            row["cell"]
        )
        print("-" * 100)

        print(row["source"])

else:

    print(
        "No notebook cells indicating per-run "
        "prediction saving were found."
    )


# ============================================================
# 5. SEARCH NOTEBOOK OUTPUTS FOR REPEATED-RUN AUROCs
# ============================================================

print("\n" + "=" * 100)
print("REPEATED-RUN METRIC OUTPUT DISCOVERY")
print("=" * 100)

metric_hits = []

for nb_path in ROOT.rglob("*.ipynb"):

    try:

        nb = json.loads(
            nb_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

        for cell_idx, cell in enumerate(
            nb.get("cells", [])
        ):

            parts = []

            parts.extend(
                cell.get("source", [])
            )

            for out in cell.get(
                "outputs", []
            ):

                if "text" in out:
                    parts.extend(out["text"])

                if "data" in out:

                    data = out["data"]

                    if "text/plain" in data:
                        parts.extend(
                            data["text/plain"]
                        )

            txt = "".join(parts)

            low = txt.lower()

            # Candidate repeated-run output
            if (
                ("run 1/5" in low or
                 "run 1" in low)
                and
                ("auc" in low)
            ):

                metric_hits.append({
                    "notebook": nb_path.name,
                    "cell": cell_idx,
                    "context": txt[:7000]
                })

    except Exception:
        pass


if metric_hits:

    metric_hits_df = pd.DataFrame(
        metric_hits
    )

    print(
        f"Repeated-run output cells found: "
        f"{len(metric_hits_df)}"
    )

    for _, row in metric_hits_df.iterrows():

        print("\n" + "-" * 100)
        print(
            "NOTEBOOK:",
            row["notebook"]
        )
        print(
            "CELL    :",
            row["cell"]
        )
        print("-" * 100)

        print(row["context"])

else:

    print(
        "No repeated-run metric output "
        "cells found."
    )


# ============================================================
# 6. IMPORTANT EXISTING TRACEABILITY SUMMARY
# ============================================================

print("\n" + "=" * 100)
print("KNOWN TRACEABILITY FROM CELL 5C")
print("=" * 100)

print("""
Initial COST:
  5-run AUROCs:
    Run 1 = 0.5962
    Run 2 = 0.6496  <-- selected best run
    Run 3 = 0.5906
    Run 4 = 0.6269
    Run 5 = 0.6036

  Reported repeated-run summary:
    AUROC = 0.613 ± 0.022

  Saved patient-level prediction:
    cost_before_predictions.npz

  Saved prediction corresponds to:
    Run 2 / best-performing run

Improved COST saved prediction:
    cost_improved_predictions.npz

  Single AUROC   = 0.8002
  Ensemble AUROC = 0.8025
""")


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 100)
print("CELL 5D COMPLETE")
print("=" * 100)

print("""
PURPOSE OF THIS CELL:

Determine whether patient-level predictions for EACH
repeated run still exist.

IF PER-RUN PREDICTIONS EXIST:
  -> We can perform a proper between-run statistical audit
     without retraining.

IF PER-RUN PREDICTIONS DO NOT EXIST:
  -> Do NOT retrain automatically.
  -> Use existing repeated-run AUROC distributions plus
     transparent representative-run traceability in the
     reviewer response.

NO TRAINING
NO MODEL INFERENCE
NO DELONG
NO BOOTSTRAP
NO RESULTS MODIFIED
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 6 — REVIEWER ABLATION ARTIFACT AUDIT
#
# PURPOSE:
# Find ALL existing ablation results before running anything.
#
# Search for:
#   - ablation configurations
#   - class-wise metrics
#   - AP / mAP
#   - AUROC / AUPRC
#   - sensitivity / specificity
#   - precision / recall / F1
#   - gating / salience / attention / BiLSTM
#
# NO TRAINING
# NO INFERENCE
# NO NEW METRIC CALCULATION
# NO RESULTS MODIFIED
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import os

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

print("=" * 100)
print("REVIEWER — ABLATION / CLASS-WISE METRIC ARTIFACT AUDIT")
print("=" * 100)

assert ROOT.exists(), f"ROOT not found: {ROOT}"

# ============================================================
# 1. SEARCH FILENAMES
# ============================================================

keywords = [
    "ablation",
    "classwise",
    "class_wise",
    "class-wise",
    "map",
    "average_precision",
    "precision",
    "recall",
    "f1",
    "auc",
    "auroc",
    "auprc",
    "sensitivity",
    "specificity",
    "gating",
    "salience",
    "attention",
    "bilstm",
    "component",
]

extensions = {
    ".csv", ".xlsx", ".npz", ".npy",
    ".pkl", ".pickle", ".json"
}

file_rows = []

for p in ROOT.rglob("*"):

    if not p.is_file():
        continue

    if p.suffix.lower() not in extensions:
        continue

    low = str(p).lower()

    matched = [
        k for k in keywords
        if k in low
    ]

    if matched:

        file_rows.append({
            "filename": p.name,
            "extension": p.suffix.lower(),
            "matched": ", ".join(matched),
            "size_kb": p.stat().st_size / 1024,
            "path": str(p)
        })

files_df = pd.DataFrame(file_rows)

print("\n" + "=" * 100)
print("CANDIDATE ABLATION / METRIC FILES")
print("=" * 100)

if len(files_df):

    files_df = (
        files_df
        .sort_values(["filename", "path"])
        .reset_index(drop=True)
    )

    display(files_df)

else:
    print("No filename-level candidates found.")


# ============================================================
# 2. INSPECT ALL REVISION CSV FILES
# ============================================================

print("\n" + "=" * 100)
print("REVISION CSV CONTENT AUDIT")
print("=" * 100)

csv_candidates = []

for p in ROOT.rglob("*.csv"):

    low = str(p).lower()

    # Focus on revision/results/table directories first
    if not any(
        x in low
        for x in [
            "revision",
            "result",
            "saved_prediction",
            "output",
            "table"
        ]
    ):
        continue

    try:

        df = pd.read_csv(p)

        cols_lower = [
            str(c).lower()
            for c in df.columns
        ]

        joined_cols = " ".join(cols_lower)

        relevant_terms = [
            "configuration",
            "ablation",
            "auc",
            "auroc",
            "auprc",
            "precision",
            "recall",
            "f1",
            "sensitivity",
            "specificity",
            "map",
            "sparsity",
            "gating",
            "salience",
            "attention"
        ]

        matched = [
            k for k in relevant_terms
            if k in joined_cols
        ]

        if matched:

            csv_candidates.append({
                "path": str(p),
                "filename": p.name,
                "shape": df.shape,
                "columns": list(df.columns),
                "matched": matched
            })

    except Exception:
        pass


print(
    f"Relevant CSV tables found: "
    f"{len(csv_candidates)}"
)

for item in csv_candidates:

    print("\n" + "-" * 100)
    print("FILE   :", item["filename"])
    print("PATH   :", item["path"])
    print("SHAPE  :", item["shape"])
    print("MATCH  :", item["matched"])
    print("COLUMNS:", item["columns"])

    try:
        df = pd.read_csv(item["path"])
        display(df)
    except Exception as e:
        print("Could not display:", repr(e))


# ============================================================
# 3. SEARCH NOTEBOOKS FOR ABLATION TABLES / OUTPUTS
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK ABLATION SEARCH")
print("=" * 100)

search_terms = [
    "ablation",
    "without salience",
    "without attention",
    "without gating",
    "gating off",
    "gating on",
    "bilstm",
    "bi-lstm",
    "class-wise",
    "classwise",
    "average precision",
    "mAP",
]

notebook_hits = []

for nb_path in ROOT.rglob("*.ipynb"):

    try:

        nb = json.loads(
            nb_path.read_text(
                encoding="utf-8",
                errors="ignore"
            )
        )

        for cell_idx, cell in enumerate(
            nb.get("cells", [])
        ):

            parts = []

            parts.extend(
                cell.get("source", [])
            )

            for out in cell.get(
                "outputs", []
            ):

                if "text" in out:
                    parts.extend(out["text"])

                if "data" in out:

                    data = out["data"]

                    if "text/plain" in data:
                        parts.extend(
                            data["text/plain"]
                        )

            text = "".join(parts)
            low = text.lower()

            matched = [
                term
                for term in search_terms
                if term.lower() in low
            ]

            if matched:

                notebook_hits.append({
                    "notebook": nb_path.name,
                    "cell": cell_idx,
                    "matched": ", ".join(matched),
                    "context": text[:6000]
                })

    except Exception:
        pass


print(
    f"Candidate notebook cells: "
    f"{len(notebook_hits)}"
)

if notebook_hits:

    nb_hits_df = pd.DataFrame(
        notebook_hits
    )

    for _, row in nb_hits_df.iterrows():

        print("\n" + "-" * 100)
        print("NOTEBOOK :", row["notebook"])
        print("CELL     :", row["cell"])
        print("MATCHED  :", row["matched"])
        print("-" * 100)

        print(row["context"])


# ============================================================
# 4. SEARCH FOR CONFIGURATION-LEVEL RESULTS
# ============================================================

print("\n" + "=" * 100)
print("CONFIGURATION-LEVEL RESULT SEARCH")
print("=" * 100)

config_terms = [
    "full model",
    "improved cost",
    "initial cost",
    "without",
    "no salience",
    "no attention",
    "no fusion",
    "gating off",
    "gating on",
    "unidirectional",
    "bidirectional",
]

config_hits = []

for p in ROOT.rglob("*.csv"):

    try:

        df = pd.read_csv(p)

        if len(df) == 0:
            continue

        text = (
            " ".join(
                df.astype(str)
                  .fillna("")
                  .values
                  .flatten()
            )
        ).lower()

        matched = [
            term
            for term in config_terms
            if term in text
        ]

        if matched:

            config_hits.append({
                "filename": p.name,
                "matched": ", ".join(matched),
                "shape": df.shape,
                "path": str(p)
            })

    except Exception:
        pass


if config_hits:

    config_df = pd.DataFrame(
        config_hits
    )

    display(config_df)

    for _, row in config_df.iterrows():

        print("\n" + "-" * 100)
        print("FILE:", row["path"])
        print("-" * 100)

        try:
            display(pd.read_csv(row["path"]))
        except Exception:
            pass

else:
    print(
        "No configuration-level CSV "
        "results found."
    )


# ============================================================
# 5. SEARCH NPZ FILES FOR LABELS + PREDICTIONS
# ============================================================

print("\n" + "=" * 100)
print("ABLATION PREDICTION NPZ AUDIT")
print("=" * 100)

npz_rows = []

for p in ROOT.rglob("*.npz"):

    low = str(p).lower()

    if not any(
        x in low
        for x in [
            "ablation",
            "gating",
            "salience",
            "attention",
            "bilstm",
            "cost"
        ]
    ):
        continue

    try:

        z = np.load(
            p,
            allow_pickle=True
        )

        keys = z.files

        npz_rows.append({
            "filename": p.name,
            "keys": keys,
            "path": str(p)
        })

        print("\n" + "-" * 100)
        print("FILE:", p)
        print("KEYS:", keys)

        for key in keys:

            arr = np.asarray(z[key])

            print(
                f"  {key:30s} "
                f"shape={arr.shape} "
                f"dtype={arr.dtype}"
            )

    except Exception as e:
        pass


# ============================================================
# 6. FINAL STATUS
# ============================================================

print("\n" + "=" * 100)
print("CELL 6 COMPLETE")
print("=" * 100)

print("""
✓ Existing ablation artifacts searched
✓ Existing metric tables inspected
✓ Notebook ablation outputs searched
✓ Configuration-level results searched
✓ Existing ablation prediction files audited

NO TRAINING
NO MODEL INFERENCE
NO NEW METRICS
NO RESULTS MODIFIED

NEXT:
Determine exactly which reviewer-requested ablation metrics
already exist and which, if any, genuinely require one
targeted calculation.
""")

print("=" * 100)

In [ ]:
# ============================================================
# CELL 7 — SELF-CONTAINED COST IMPROVED FLOPs / MACs
# Loads exact frozen DSRNNv2 checkpoint automatically
# NO TRAINING / NO FINE-TUNING
# ============================================================

import sys, subprocess, math
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd

print("=" * 100)
print("COST IMPROVED — SELF-CONTAINED FLOPs / MACs AUDIT")
print("=" * 100)

# ============================================================
# 1. THOP
# ============================================================

try:
    from thop import profile
    print("✓ THOP already installed")
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "thop"
    ])
    from thop import profile
    print("✓ THOP installed")


# ============================================================
# 2. LOCATE CHECKPOINT
# ============================================================

ROOT = Path("/content/drive/MyDrive/COST_PROJECT")

CHECKPOINT = (
    ROOT /
    "mimic-iv-2.1/saved_models/dsrnn_v2_best.pt"
)

assert CHECKPOINT.exists(), (
    f"Checkpoint not found:\n{CHECKPOINT}"
)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device     :", DEVICE)
print("Checkpoint :", CHECKPOINT)


# ============================================================
# 3. EXACT COST IMPROVED ARCHITECTURE
# ============================================================

class SinusoidalPE(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=256,
        dropout=0.1
    ):
        super().__init__()

        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)

        pos = (
            torch.arange(max_len)
            .unsqueeze(1)
            .float()
        )

        div = torch.exp(
            torch.arange(
                0, d_model, 2
            ).float()
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return self.dropout(
            x + self.pe[:, :x.size(1)]
        )


class StochasticDepth(nn.Module):

    def __init__(self, p=0.1):

        super().__init__()
        self.p = p

    def forward(self, x, residual):

        if not self.training or self.p == 0:
            return x + residual

        keep = torch.bernoulli(
            torch.tensor(
                1 - self.p,
                device=x.device
            )
        ).item()

        return x + residual * keep


class ClinicalSalienceV2(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        n_features
    ):

        super().__init__()

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim * 2
        )

        self.norm = nn.LayerNorm(
            hidden_dim
        )

        self.fc2 = nn.Linear(
            hidden_dim,
            n_features
        )

        self.tau = nn.Parameter(
            torch.tensor(0.5)
        )

    def forward(
        self,
        x_t,
        h_prev
    ):

        h = self.fc1(
            torch.cat(
                [x_t, h_prev],
                -1
            )
        )

        h, gate = h.chunk(
            2,
            -1
        )

        h = self.norm(
            h * torch.sigmoid(gate)
        )

        s = torch.sigmoid(
            self.fc2(h)
        )

        hard = (
            s > self.tau
        ).float()

        return (
            hard - s.detach() + s,
            s
        )


class TemporalMultiHeadAttn(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        dropout=0.1
    ):

        super().__init__()

        self.attn = nn.MultiheadAttention(
            d_model,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            d_model
        )

        self.sdrop = StochasticDepth(
            0.1
        )

    def forward(self, x):

        out, w = self.attn(
            x, x, x
        )

        return (
            self.norm(
                self.sdrop(
                    x,
                    out
                )
            ),
            w
        )


class StaticTemporalFusion(nn.Module):

    def __init__(
        self,
        d_temporal,
        n_static,
        d_out,
        n_heads=4,
        dropout=0.1
    ):

        super().__init__()

        self.static_proj = nn.Linear(
            n_static,
            d_temporal
        )

        self.cross_attn = nn.MultiheadAttention(
            d_temporal,
            n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(
            d_temporal
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                d_temporal * 2,
                d_temporal
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(
                d_temporal,
                d_out
            )
        )

    def forward(
        self,
        s,
        h
    ):

        q = self.static_proj(
            s
        ).unsqueeze(1)

        ctx, w = self.cross_attn(
            q,
            h,
            h
        )

        ctx = self.norm(
            ctx.squeeze(1)
        )

        z = self.ffn(
            torch.cat(
                [
                    ctx,
                    h.mean(1)
                ],
                -1
            )
        )

        return z, w.squeeze(1)


class DSRNNv2(nn.Module):

    def __init__(
        self,
        n_input=93,
        n_static=11,
        hidden_size=256,
        n_lstm_layers=2,
        n_heads=4,
        attention_dim=128,
        salience_hidden=128,
        dropout=0.3,
        lambda_sparse=5e-4
    ):

        super().__init__()

        self.hidden_size = hidden_size
        self.lambda_sparse = lambda_sparse

        self.input_proj = nn.Sequential(
            nn.Linear(
                n_input,
                hidden_size
            ),
            nn.LayerNorm(
                hidden_size
            ),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.pos_enc = SinusoidalPE(
            hidden_size,
            dropout=dropout * 0.5
        )

        self.salience = ClinicalSalienceV2(
            hidden_size * 2,
            salience_hidden,
            hidden_size
        )

        self.lstm = nn.LSTM(
            hidden_size,
            hidden_size,
            num_layers=n_lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )

        self.bidir_proj = nn.Sequential(
            nn.Linear(
                hidden_size * 2,
                hidden_size
            ),
            nn.LayerNorm(
                hidden_size
            ),
            nn.GELU()
        )

        self.temporal_attn = (
            TemporalMultiHeadAttn(
                hidden_size,
                n_heads,
                dropout
            )
        )

        self.fusion = StaticTemporalFusion(
            hidden_size,
            n_static,
            attention_dim,
            n_heads,
            dropout
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(
                attention_dim
            ),
            nn.Dropout(dropout),
            nn.Linear(
                attention_dim,
                attention_dim // 2
            ),
            nn.GELU(),
            nn.Dropout(
                dropout * 0.5
            ),
            nn.Linear(
                attention_dim // 2,
                1
            )
        )

    def forward(
        self,
        x_temporal,
        x_static
    ):

        b, seq, _ = x_temporal.shape

        xp = self.pos_enc(
            self.input_proj(
                x_temporal
            )
        )

        hp = torch.zeros(
            b,
            self.hidden_size,
            device=x_temporal.device
        )

        gs = []

        sp = torch.tensor(
            0.0,
            device=x_temporal.device
        )

        for t in range(seq):

            xt = xp[:, t]

            m, _ = self.salience(
                xt,
                hp
            )

            xg = (
                m * xt +
                (1 - m) * xt * 0.1
            )

            gs.append(
                xg.unsqueeze(1)
            )

            hp = (
                hp * 0.9 +
                xg.detach() * 0.1
            )

            sp += m.norm(p=1)

        g = torch.cat(
            gs,
            1
        )

        sp = (
            self.lambda_sparse *
            sp /
            (seq * b)
        )

        lo, _ = self.lstm(g)

        lo = self.bidir_proj(lo)

        H, _ = self.temporal_attn(
            lo
        )

        z, _ = self.fusion(
            x_static,
            H
        )

        return (
            self.classifier(z)
            .squeeze(-1),
            sp
        )


# ============================================================
# 4. LOAD FROZEN CHECKPOINT
# ============================================================

ckpt = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

cfg = ckpt.get(
    "config",
    {}
)

model = DSRNNv2(
    n_input=93,
    n_static=11,
    hidden_size=cfg.get(
        "hidden_size", 256
    ),
    n_lstm_layers=cfg.get(
        "n_lstm_layers", 2
    ),
    n_heads=cfg.get(
        "n_heads", 4
    ),
    attention_dim=cfg.get(
        "attention_dim", 128
    ),
    salience_hidden=cfg.get(
        "salience_hidden", 128
    ),
    dropout=cfg.get(
        "dropout", 0.3
    ),
    lambda_sparse=cfg.get(
        "lambda_sparse", 5e-4
    )
).to(DEVICE)

load_result = model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

model.eval()

params = sum(
    p.numel()
    for p in model.parameters()
)

print("\nCheckpoint load:")
print(load_result)
print("Parameters :", f"{params:,}")
print("Eval mode  :", not model.training)

assert params == 3_653_890


# ============================================================
# 5. ONE-PATIENT INPUT
# ============================================================

x_temporal = torch.zeros(
    1, 48, 93,
    device=DEVICE
)

x_static = torch.zeros(
    1, 11,
    device=DEVICE
)

print("\nInput:")
print(
    "Temporal :",
    tuple(x_temporal.shape)
)
print(
    "Static   :",
    tuple(x_static.shape)
)


# ============================================================
# 6. PROFILE
# ============================================================

print(
    "\nProfiling computational complexity..."
)

with torch.no_grad():

    macs, thop_params = profile(
        model,
        inputs=(
            x_temporal,
            x_static
        ),
        verbose=False
    )

macs = float(macs)

# conventional approximation
flops = 2.0 * macs

MMAC = macs / 1e6
MFLOPS = flops / 1e6

print("\n" + "=" * 100)
print("RESULT")
print("=" * 100)

print(
    f"Parameters           : "
    f"{params:,}"
)

print(
    f"THOP parameters      : "
    f"{thop_params:,.0f}"
)

print(
    f"MACs / patient       : "
    f"{MMAC:.3f} MMAC"
)

print(
    f"Approx FLOPs/patient : "
    f"{MFLOPS:.3f} MFLOPs"
)


# ============================================================
# 7. SAVE
# ============================================================

result = pd.DataFrame({
    "Metric": [
        "Parameters",
        "Model Memory (MB)",
        "Gating OFF Latency (ms/patient)",
        "Gating ON Latency (ms/patient)",
        "Gating ON Sparsity (%)",
        "MACs (MMAC/patient)",
        "Approx FLOPs (MFLOPs/patient)"
    ],
    "COST Improved": [
        f"{params:,}",
        "13.94",
        "0.102 ± 0.001",
        "0.416 ± 0.006",
        "99.01",
        f"{MMAC:.3f}",
        f"{MFLOPS:.3f}"
    ]
})

print("\nREVIEWER-READY SUMMARY")
display(result)

OUT_DIR = (
    ROOT /
    "mimic-iv-2.1/revision2_outputs"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUT = (
    OUT_DIR /
    "cost_improved_flops_audit.csv"
)

result.to_csv(
    OUT,
    index=False
)

print("\nSaved:", OUT)

print("\n" + "=" * 100)
print("✓ CELL 7 COMPLETE")
print("✓ EXACT CHECKPOINT LOADED WITH strict=True")
print("✓ NO TRAINING")
print("✓ NO FINE-TUNING")
print("=" * 100)

In [ ]:
# ============================================================
# CELL 8 — CORRECTED FULL COST IMPROVED FLOPs AUDIT
#
# Corrects THOP undercount by explicitly accounting for:
#   - Linear layers
#   - Clinical Salience repeated over 48 time steps
#   - 2-layer bidirectional LSTM
#   - Temporal Multi-Head Attention
#   - Static–Temporal Cross Attention
#   - Fusion FFN
#   - Classifier
#
# Reports MACs and FLOPs separately.
#
# NO TRAINING
# NO INFERENCE ON PATIENT DATA
# NO WEIGHT UPDATE
# ============================================================

import math
import pandas as pd
from pathlib import Path

print("=" * 100)
print("COST IMPROVED — CORRECTED FULL COMPUTATIONAL COMPLEXITY")
print("=" * 100)

# ============================================================
# ARCHITECTURE
# ============================================================

T = 48
D_IN = 93
D = 256
STATIC = 11

LSTM_H = 256
LSTM_LAYERS = 2
DIRECTIONS = 2

ATT_OUT = 128

N_PARAMS = 3_653_890


# ============================================================
# HELPER
# ============================================================

def linear_macs(n_tokens, din, dout):
    """Matrix multiplication MACs for a Linear layer."""
    return n_tokens * din * dout


# ============================================================
# 1. INPUT PROJECTION
# 93 -> 256 for 48 time steps
# ============================================================

input_proj = linear_macs(
    T,
    D_IN,
    D
)


# ============================================================
# 2. CLINICAL SALIENCE
#
# Every time step:
#
# input to fc1:
#   concat(x_t, h_prev)
#   256 + 256 = 512
#
# fc1:
#   512 -> 256
#
# because implementation:
# hidden_dim=128
# fc1 output = hidden_dim*2 = 256
#
# fc2:
#   128 -> 256
#
# repeated for T=48
# ============================================================

salience_fc1 = linear_macs(
    T,
    512,
    256
)

salience_fc2 = linear_macs(
    T,
    128,
    256
)

salience_total = (
    salience_fc1 +
    salience_fc2
)


# ============================================================
# 3. BIDIRECTIONAL LSTM
#
# PyTorch LSTM:
#
# Four gates:
# MACs per time step/direction =
#
# 4 * H * (input_size + H)
#
# Layer 0:
# input_size = 256
#
# Layer 1:
# input_size = 512
# because bidirectional output from layer 0
# ============================================================

lstm_layer0 = (
    T *
    DIRECTIONS *
    4 *
    LSTM_H *
    (D + LSTM_H)
)

lstm_layer1 = (
    T *
    DIRECTIONS *
    4 *
    LSTM_H *
    ((2 * LSTM_H) + LSTM_H)
)

lstm_total = (
    lstm_layer0 +
    lstm_layer1
)


# ============================================================
# 4. BIDIRECTIONAL PROJECTION
# 512 -> 256 × 48
# ============================================================

bidir_proj = linear_macs(
    T,
    512,
    256
)


# ============================================================
# 5. TEMPORAL MULTI-HEAD SELF ATTENTION
#
# Sequence length L=48
# embedding D=256
#
# Q,K,V projections:
#   3 * L * D * D
#
# attention score QK^T:
#   L * L * D
#
# attention × V:
#   L * L * D
#
# output projection:
#   L * D * D
# ============================================================

temporal_qkv = (
    3 *
    T *
    D *
    D
)

temporal_scores = (
    T *
    T *
    D
)

temporal_weighted_values = (
    T *
    T *
    D
)

temporal_output_proj = (
    T *
    D *
    D
)

temporal_attention = (
    temporal_qkv +
    temporal_scores +
    temporal_weighted_values +
    temporal_output_proj
)


# ============================================================
# 6. STATIC PROJECTION
# 11 -> 256
# ============================================================

static_proj = (
    STATIC *
    D
)


# ============================================================
# 7. STATIC-TEMPORAL CROSS ATTENTION
#
# Query length = 1
# Key/value length = 48
#
# PyTorch MultiheadAttention still applies:
#
# Q projection:
#   1 × D × D
#
# K projection:
#   T × D × D
#
# V projection:
#   T × D × D
#
# QK^T:
#   1 × T × D
#
# attention × V:
#   1 × T × D
#
# output projection:
#   1 × D × D
# ============================================================

cross_q = (
    D * D
)

cross_k = (
    T *
    D *
    D
)

cross_v = (
    T *
    D *
    D
)

cross_scores = (
    T *
    D
)

cross_weighted_values = (
    T *
    D
)

cross_output_proj = (
    D *
    D
)

cross_attention = (
    cross_q +
    cross_k +
    cross_v +
    cross_scores +
    cross_weighted_values +
    cross_output_proj
)


# ============================================================
# 8. FUSION FFN
#
# concat:
# context 256 + mean temporal 256 = 512
#
# 512 -> 256 -> 128
# ============================================================

fusion_ffn = (
    512 * 256 +
    256 * 128
)


# ============================================================
# 9. CLASSIFIER
#
# LayerNorm 128
# Linear 128 -> 64
# Linear 64 -> 1
# ============================================================

classifier = (
    128 * 64 +
    64 * 1
)


# ============================================================
# 10. TOTAL MACs
# ============================================================

components = {
    "Input projection": input_proj,
    "Clinical salience": salience_total,
    "BiLSTM layer 0": lstm_layer0,
    "BiLSTM layer 1": lstm_layer1,
    "Bidirectional projection": bidir_proj,
    "Temporal self-attention": temporal_attention,
    "Static projection": static_proj,
    "Cross-attention": cross_attention,
    "Fusion FFN": fusion_ffn,
    "Classifier": classifier,
}

TOTAL_MACS = sum(
    components.values()
)

# Standard convention:
# 1 MAC = multiply + accumulate ≈ 2 FLOPs

TOTAL_FLOPS = (
    2 * TOTAL_MACS
)


# ============================================================
# 11. COMPONENT TABLE
# ============================================================

rows = []

for name, macs in components.items():

    rows.append({
        "Component": name,
        "MACs": int(macs),
        "MMACs": macs / 1e6,
        "Approx MFLOPs": (2 * macs) / 1e6,
        "Share (%)": (
            100 * macs / TOTAL_MACS
        )
    })

component_df = pd.DataFrame(rows)

print("\n" + "=" * 100)
print("COMPONENT-WISE COMPLEXITY")
print("=" * 100)

display(
    component_df.style.format({
        "MACs": "{:,.0f}",
        "MMACs": "{:.3f}",
        "Approx MFLOPs": "{:.3f}",
        "Share (%)": "{:.2f}"
    })
)


# ============================================================
# 12. FINAL RESULT
# ============================================================

print("\n" + "=" * 100)
print("FULL COST IMPROVED COMPLEXITY")
print("=" * 100)

print(
    f"Parameters             : "
    f"{N_PARAMS:,}"
)

print(
    f"Total MACs / patient   : "
    f"{TOTAL_MACS:,.0f}"
)

print(
    f"Total MMACs / patient  : "
    f"{TOTAL_MACS/1e6:.3f}"
)

print(
    f"Total GMACs / patient  : "
    f"{TOTAL_MACS/1e9:.6f}"
)

print()

print(
    f"Approx FLOPs / patient : "
    f"{TOTAL_FLOPS:,.0f}"
)

print(
    f"Approx MFLOPs/patient  : "
    f"{TOTAL_FLOPS/1e6:.3f}"
)

print(
    f"Approx GFLOPs/patient  : "
    f"{TOTAL_FLOPS/1e9:.6f}"
)


# ============================================================
# 13. COMPARE AGAINST THOP RESULT
# ============================================================

THOP_MACS = 142.263e6

difference = (
    TOTAL_MACS -
    THOP_MACS
)

print("\n" + "=" * 100)
print("THOP CROSS-CHECK")
print("=" * 100)

print(
    f"THOP MACs              : "
    f"{THOP_MACS/1e6:.3f} MMAC"
)

print(
    f"Corrected manual MACs  : "
    f"{TOTAL_MACS/1e6:.3f} MMAC"
)

print(
    f"Difference             : "
    f"{difference/1e6:.3f} MMAC"
)

print(
    f"THOP coverage          : "
    f"{100*THOP_MACS/TOTAL_MACS:.2f}%"
)


# ============================================================
# 14. REVIEWER-READY COMPUTATIONAL TABLE
# ============================================================

reviewer_table = pd.DataFrame({
    "Metric": [
        "Parameters",
        "Model memory (FP32, MB)",
        "Computational cost (MMAC/patient)",
        "Approx. computational cost (MFLOPs/patient)",
        "Inference latency — Gating OFF (ms/patient)",
        "Inference latency — Gating ON (ms/patient)",
        "Relative latency",
        "Gating ON sparsity (%)"
    ],

    "COST Improved": [
        f"{N_PARAMS:,}",
        "13.94",
        f"{TOTAL_MACS/1e6:.3f}",
        f"{TOTAL_FLOPS/1e6:.3f}",
        "0.102 ± 0.001",
        "0.416 ± 0.006",
        "4.06×",
        "99.01"
    ]
})

print("\n" + "=" * 100)
print("REVIEWER-READY COMPUTATIONAL SUMMARY")
print("=" * 100)

display(reviewer_table)


# ============================================================
# 15. SAVE
# ============================================================

ROOT = Path(
    "/content/drive/MyDrive/COST_PROJECT"
)

OUT_DIR = (
    ROOT /
    "mimic-iv-2.1/revision2_outputs"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

component_file = (
    OUT_DIR /
    "cost_improved_flops_components.csv"
)

summary_file = (
    OUT_DIR /
    "cost_improved_computational_final.csv"
)

component_df.to_csv(
    component_file,
    index=False
)

reviewer_table.to_csv(
    summary_file,
    index=False
)

print("\nSaved:")
print(component_file)
print(summary_file)


print("\n" + "=" * 100)
print("✓ CELL 8 COMPLETE")
print("✓ FULL ARCHITECTURE ACCOUNTED FOR")
print("✓ ATTENTION INCLUDED EXPLICITLY")
print("✓ BiLSTM INCLUDED EXPLICITLY")
print("✓ SALIENCE INCLUDED EXPLICITLY")
print("✓ NO TRAINING")
print("✓ NO MODEL MODIFICATION")
print("=" * 100)